<a href="https://colab.research.google.com/github/tskir/london-housing/blob/main/notebooks/approval_and_commencement_lag.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Planning London Datahub — approval and commencement lag

Pulls **approved** `applications` records (`valid_date` in 2021–2022, capped at 100,000 rows) and looks at two lags:

1. **Valid → decision**: `decision_date - valid_date`, i.e. how long approval took.
2. **Decision → commencement**: `actual_commencement_date - decision_date`, i.e. how long after approval building actually started.

`actual_commencement_date` is null for a large share of approved applications — most either haven't started yet or never report a start date back to the LPA. We surface null rates explicitly at each stage rather than silently dropping rows, and only compute lag statistics over the rows where both dates are present.

In [ ]:
import requests
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', 100)


## Config

In [ ]:
API_URL = "https://planningdata.london.gov.uk/api-guest"
INDEX = "applications"
HEADERS = {
    "X-API-AllowRequest": "be2rmRnt&",
    "Content-Type": "application/json",
}

PAGE_SIZE = 1000       # docs per scroll page
MAX_RECORDS = 100_000  # cap on the total number of rows pulled

FILTER_QUERY = {
    "bool": {
        "must": [
            {"term": {"status.raw": "Approved"}},
            {
                "range": {
                    "valid_date": {
                        "gte": "01/01/2021",
                        "lte": "31/12/2022",
                    }
                }
            },
        ]
    }
}

SOURCE_FIELDS = [
    "id",
    "lpa_name",
    "status",
    "valid_date",
    "decision_date",
    "actual_commencement_date",
    "application_details.intended_commencement_date",
]


## Pull the data (scroll API)

In [ ]:
def fetch_subset(index, query, source_fields, page_size, max_records, scroll_ttl="2m"):
    all_hits = []

    init_body = {
        "size": page_size,
        "query": query,
        "_source": source_fields,
    }
    resp = requests.post(
        f"{API_URL}/{index}/_search?scroll={scroll_ttl}",
        headers=HEADERS,
        json=init_body,
    )
    resp.raise_for_status()
    data = resp.json()

    total = data["hits"]["total"]["value"] if isinstance(data["hits"]["total"], dict) else data["hits"]["total"]
    print(f"Matching records available: {total:,}")

    scroll_id = data.get("_scroll_id")
    hits = data["hits"]["hits"]

    while hits:
        all_hits.extend(hits)
        if len(all_hits) >= max_records:
            all_hits = all_hits[:max_records]
            break
        resp = requests.post(
            f"{API_URL}/_search/scroll",
            headers=HEADERS,
            json={"scroll": scroll_ttl, "scroll_id": scroll_id},
        )
        resp.raise_for_status()
        data = resp.json()
        scroll_id = data.get("_scroll_id")
        hits = data["hits"]["hits"]

    print(f"Pulled {len(all_hits):,} records")
    return all_hits


raw_hits = fetch_subset(INDEX, FILTER_QUERY, SOURCE_FIELDS, PAGE_SIZE, MAX_RECORDS)


## Flatten and parse dates

In [ ]:
sources = [h["_source"] for h in raw_hits]
df = pd.json_normalize(sources, sep=".")

date_cols = [
    "valid_date",
    "decision_date",
    "actual_commencement_date",
    "application_details.intended_commencement_date",
]
for col in date_cols:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], format="%d/%m/%Y", errors="coerce")

print(df.shape)
df.head()


## Surface nulls at each stage

Before computing any lag, show how many rows actually have both dates needed for that lag — and flag rows with a negative lag (decision before valid, or commencement before decision), which usually indicate a data-entry issue rather than a real negative duration.

In [ ]:
n_total = len(df)

stage_nulls = pd.DataFrame({
    "field": ["valid_date", "decision_date", "actual_commencement_date"],
    "n_null": [df[c].isna().sum() for c in ["valid_date", "decision_date", "actual_commencement_date"]],
})
stage_nulls["pct_null"] = (stage_nulls["n_null"] / n_total * 100).round(2)

print(f"Total approved rows pulled: {n_total:,}")
display(stage_nulls)


## Lag 1: valid date → decision date

In [ ]:
has_valid_decision = df["valid_date"].notna() & df["decision_date"].notna()
n_has_both = has_valid_decision.sum()
print(f"Rows with both valid_date and decision_date: {n_has_both:,} / {n_total:,} "
      f"({n_has_both / n_total * 100:.1f}%)")

lag1 = df.loc[has_valid_decision, ["id", "lpa_name", "valid_date", "decision_date"]].copy()
lag1["lag_days"] = (lag1["decision_date"] - lag1["valid_date"]).dt.days

n_negative = (lag1["lag_days"] < 0).sum()
print(f"Rows with a negative lag (decision before valid_date, likely a data issue): {n_negative:,}")

lag1_clean = lag1[lag1["lag_days"] >= 0]
display(lag1_clean["lag_days"].describe())


In [ ]:
ax = lag1_clean["lag_days"].plot(
    kind="hist", bins=60, figsize=(10, 4),
    title=f"Days from valid_date to decision_date (n={len(lag1_clean):,})",
)
ax.set_xlabel("days")
ax.figure.tight_layout()
plt.show()


## Lag 2: decision date → commencement date

Most approved applications will not have started building yet, or never report `actual_commencement_date` back to the LPA — this is expected, not a data quality problem. The chart below only covers rows where a start date has actually been recorded; the null rate printed above it tells you what fraction of approvals that represents.

In [ ]:
has_decision_commencement = df["decision_date"].notna() & df["actual_commencement_date"].notna()
n_has_commencement = has_decision_commencement.sum()
print(f"Approved rows with a recorded actual_commencement_date: {n_has_commencement:,} / {n_total:,} "
      f"({n_has_commencement / n_total * 100:.1f}%)")
print(f"Approved rows with no commencement date yet (not started, or unreported): "
      f"{n_total - n_has_commencement:,} ({(n_total - n_has_commencement) / n_total * 100:.1f}%)")

lag2 = df.loc[has_decision_commencement, ["id", "lpa_name", "decision_date", "actual_commencement_date"]].copy()
lag2["lag_days"] = (lag2["actual_commencement_date"] - lag2["decision_date"]).dt.days

n_negative2 = (lag2["lag_days"] < 0).sum()
print(f"Rows with a negative lag (commencement before decision, likely a data issue): {n_negative2:,}")

lag2_clean = lag2[lag2["lag_days"] >= 0]
display(lag2_clean["lag_days"].describe())


In [ ]:
ax = lag2_clean["lag_days"].plot(
    kind="hist", bins=60, figsize=(10, 4),
    title=f"Days from decision_date to actual_commencement_date (n={len(lag2_clean):,} of {n_total:,} approved)",
)
ax.set_xlabel("days")
ax.figure.tight_layout()
plt.show()
